In [ ]:
from snowflake.snowpark.types import *
from snowflake.snowpark.functions import *
from snowflake.snowpark.context import get_active_session

session = get_active_session()

In [ ]:
schema = StructType([
    StructField("key", IntegerType()),
    StructField("value", StringType())])
df = session.create_dataframe(
    [(10, "old"), (10, "too_old"), (11, "old")], 
    schema=schema)

df.write.save_as_table("exercise_db.public.merge_t", mode="overwrite", table_type="temp")
target = session.table("exercise_db.public.merge_t")
target

In [ ]:

source = session.create_dataframe(
    [(10, "new"), (12, "new"), (13, "old")],
    schema=schema)

result = target.merge(source,
    (target["key"] == source["key"]) & (target["value"] == "too_old"),
    [when_not_matched().insert({"key": source["key"]})])
result
df = target.sort(col("key"), col("value"))
df

In [ ]:
target.delete(is_null("value"))
target

In [ ]:
target.update({"value": "changed"}, target["key"] == 11)
target

In [ ]:
target.drop_table()
target